# Add ComicInfo.xml to CBZ Archives

Recursively scans `TARGET_DIR` for `.cbz` files. If an archive does not contain `ComicInfo.xml`, the notebook creates metadata from filenames in this format:

`[Author] Title c001.cbz`

Existing `ComicInfo.xml` entries are left unchanged. Keep `DRY_RUN = True` to preview changes before writing to archives.

In [ ]:
from pathlib import Path
import re
import zipfile
from xml.etree import ElementTree


# --- Configuration ---
TARGET_DIR = Path(r"Z:\Misc")
FILE_GLOB = "*.cbz"
DRY_RUN = False

# Supports: c001, ch001, ch. 001, chapter 001, v01, vol. 01, and volume 01.
CHAPTER_AT_END = re.compile(
    r"\s+(?P<number>(?:c|ch\.?|chapter|v|vol\.?|volume)\s*"
    r"(?P<value>\d+(?:[-_]\d+)?))\s*$",
    re.IGNORECASE,
)


def parse_filename(filename: str) -> dict[str, str]:
    """Extract author, title, and optional chapter/volume number from a filename."""
    stem = Path(filename).stem.strip()
    author_match = re.match(r"^\[([^\]]+)\]\s*(.+)$", stem)
    if not author_match:
        raise ValueError("filename does not start with [Author]")

    author = author_match.group(1).strip()
    title = author_match.group(2).strip()
    chapter_match = CHAPTER_AT_END.search(title)
    number = ""
    if chapter_match:
        number = re.sub(r"\s+", "", chapter_match.group("value"))
        title = title[:chapter_match.start()].strip()

    if not title:
        raise ValueError("filename does not contain a title")
    return {"author": author, "title": title, "number": number}


def create_comic_info(metadata: dict[str, str]) -> bytes:
    """Create a UTF-8 ComicInfo.xml document from parsed filename metadata."""
    comic_info = ElementTree.Element("ComicInfo")
    for tag in ("Title", "Series", "Number", "Writer"):
        ElementTree.SubElement(comic_info, tag).text = metadata[
            {"Title": "title", "Series": "title", "Number": "number", "Writer": "author"}[tag]
        ]
    return ElementTree.tostring(comic_info, encoding="utf-8", xml_declaration=True)


def add_comic_info(cbz_path: Path, dry_run: bool = True) -> None:
    """Add ComicInfo.xml to one archive if it is missing."""
    with zipfile.ZipFile(cbz_path, "r") as archive:
        names = {name.lower().replace("\\", "/") for name in archive.namelist()}
        if "comicinfo.xml" in names:
            print(f"[SKIP] {cbz_path}: ComicInfo.xml already exists")
            return

    metadata = parse_filename(cbz_path.name)
    if dry_run:
        print(f"[DRY RUN] {cbz_path} -> ComicInfo.xml ({metadata})")
        return

    with zipfile.ZipFile(cbz_path, "a", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("ComicInfo.xml", create_comic_info(metadata))
    print(f"[ADDED] {cbz_path}: ComicInfo.xml")


def process_cbz_files() -> None:
    """Recursively find CBZ files and add missing ComicInfo.xml entries."""
    if not TARGET_DIR.is_dir():
        raise NotADirectoryError(f"Target directory not found: {TARGET_DIR}")

    cbz_files = sorted(path for path in TARGET_DIR.rglob(FILE_GLOB) if path.is_file())
    print(f"Found {len(cbz_files)} CBZ file(s) under {TARGET_DIR}")

    for cbz_path in cbz_files:
        try:
            add_comic_info(cbz_path, dry_run=DRY_RUN)
        except (OSError, ValueError, zipfile.BadZipFile) as exc:
            print(f"[ERROR] {cbz_path}: {exc}")


process_cbz_files()